In [ ]:
import os
import sys
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

In [ ]:
import jax

jax.config.update("jax_enable_x64", True)
jax.config.update("jax_platform_name", "cpu")

import jax.numpy as jnp
import numpy as np

from ajx.constraints import ConstraintResidual
from ajx.example_environments.dlo_attached import DLOAttached, DLOAttachedSettings
from ajx.example_graphics.application import Application
from ajx.example_graphics.environment_scene import EnvironmentScene
from ajx.simulation import SimulationSettings, Solver

N_BODIES = 25

def setup_dlo_environment(timestep, pgs_iterations, solver, youngs_modulus, poission_ratio, mass_density):

    env = DLOAttached(
        sim_settings=SimulationSettings(
            timestep, True, solver, pgs_iterations
        ),
        env_settings=DLOAttachedSettings(
            n_bodies=N_BODIES,
            body_length=0.03,
            mass_density=mass_density,
            constraint_residual=ConstraintResidual.BEND_TWIST.value,
            hinge_motor_attachment=False,
            body_side_length=0.06
        ),
    )
    
    shear_modulus = youngs_modulus / (2 * (1 + poission_ratio))
    linear_stiffness, bend_stiffness, torsion_stiffness = env.get_stiffness_from_material_parameters(youngs_modulus, shear_modulus)

    yz_linear_stiffness = linear_stiffness
    x_linear_stiffness = linear_stiffness
    bend_linear_stiffness = bend_stiffness
    torsion_linear_stiffness = torsion_stiffness

    env_param = env.default_param.tree_replace(
        src={
            "sparse_param.coupled_constraint_param": {
                "linear_stiffness.data": jnp.array(
                    [
                        x_linear_stiffness,
                        yz_linear_stiffness,
                        yz_linear_stiffness,
                        bend_linear_stiffness,
                        bend_linear_stiffness,
                        torsion_linear_stiffness,
                    ]
                ),
                "is_velocity": jnp.array([0, 0, 0, 0, 0, 0], dtype=bool),
            }
        }
    )

    return env, env_param


def simulate_dlo(settings_dict, tmax=None):
    """
    INPUTS:
        settings_dict: A python dict with settings.
        tmax (Optional): Stop simulation if t > tmax
    OUTPUTS:
        dlo_x: x-position data for DLO. array of size (horizon+1) x N_BODIES
        dlo_z: z_position data for DLO. array of size (horizon+1) x N_BODIES
        dz: z_position data for end point of DLO. array of size (horizon+1) x 1
    """

    # To extract simulation settings/arguments
    horizon = settings_dict["horizon"]
    timestep = settings_dict["timestep"]
    pgs_iterations = settings_dict["pgs_iterations"]
    solver = settings_dict["solver"]
    youngs_modulus = settings_dict["youngs_modulus"]
    poisson_ratio = settings_dict["poisson_ratio"]
    mass_density = settings_dict["mass_density"]

    # To setup and initialize a dlo environment
    env, env_param = setup_dlo_environment(timestep, pgs_iterations, solver, youngs_modulus, poisson_ratio, mass_density)

    state = env.state_from_angles(env_param)
    env_step = jax.jit(env.step)
    u = np.zeros([horizon,])

    # To prepare storage of rigid body data
    dlo_pos_x = [state.conf.pos[:,0]]
    dlo_pos_z = [state.conf.pos[:,2]]
    loose_end_id = env_param.rigid_body_param.names.index(f"body{N_BODIES-1}")

    # Simulation loop
    for j in range(horizon):

        if tmax is not None and (j+1)*timestep > tmax:
            break

        # Step the environment and store the observation
        state, observations = env_step(state, u, env_param)
        dlo_pos_x.append(state.conf.pos[:,0])
        dlo_pos_z.append(state.conf.pos[:,2])

    dlo_x = np.array(dlo_pos_x)
    dlo_z = np.array(dlo_pos_z)
    dz = dlo_z[:,loose_end_id]
    return dlo_x, dlo_z, dz

## Use the Direct solver as ground truth

In [ ]:
settings_dict = {
    "horizon": 1000,
    "timestep": 0.01,
    "pgs_iterations": None,
    "solver": Solver.DENSE_LINEAR,
    "youngs_modulus": 1e7,
    "poisson_ratio": 0.3,
    "mass_density": 1000.0,
}

dlo_x, dlo_z, dz = simulate_dlo(settings_dict)


# To store results for plotting 
dlo_position = {"DIRECT_SOLVER": {"x": dlo_x, "z": dlo_z, "timestep": settings_dict["timestep"]}}
dlo_dz = {"DIRECT_SOLVER": dz}
time = np.array([k*settings_dict["timestep"] for k in range(settings_dict["horizon"]+1)])

jax.clear_caches()  # Workaround

## Use the Iterative Solver for different number of iterations

In [ ]:
NUMBER_OF_ITERATIONS = (800, 1200)

settings_dict["solver"] = Solver.SPARSE_PGS

dlo_pos_x = []
dlo_pos_z = []
delta_z = []


for Nit in NUMBER_OF_ITERATIONS:
    
    settings_dict["pgs_iterations"] = Nit
    dlo_x, dlo_z, dz = simulate_dlo(settings_dict)

    dlo_pos_x.append(dlo_x)
    dlo_pos_z.append(dlo_z)
    delta_z.append(dz)


    jax.clear_caches()  # Workaround

# To store results for plotting
dlo_position["ITERATIVE_SOLVER"] = {"x": np.array(dlo_pos_x), "z": np.array(dlo_pos_z), "timestep": np.array([settings_dict["timestep"]] * len(NUMBER_OF_ITERATIONS)), "number_of_iterations": NUMBER_OF_ITERATIONS}
dlo_dz["ITERATIVE_SOLVER"] = np.array(delta_z)

In [ ]:
from matplotlib import pyplot as plt
from matplotlib.animation import FuncAnimation
import matplotlib as mpl
from IPython.display import HTML

def visualize_dlo(time, dlo_position, dz):
    """
    INPUTS:
        dlo_position: A python dict with DLO-data for the different solvers to plot.
        dz: A python python dict with dz-data for the different solvers to plot.
    OUTPUT:
        anim: Animation object to display
    """
    # To precompute axis limits
    x_direct = dlo_position["DIRECT_SOLVER"]["x"]
    z_direct = dlo_position["DIRECT_SOLVER"]["z"]
    x_min = np.min([x_direct.min()])
    x_max = np.max([x_direct.max()])
    z_min = np.min([z_direct.min()])
    z_max = np.max([z_direct.max()])

    # Make a figure and use it for an animation
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ax[0].set_xlabel("x")
    ax[0].set_ylabel("z")
    ax[0].set_xlim([x_min-0.05, x_max + 0.05])
    ax[0].set_ylim([z_min-0.2, z_max + 0.2])
    ax[0].grid(True)
    ax[0].set_aspect('equal')

    ax[1].set_xlabel("time [s]")
    ax[1].set_ylabel("dz [m]")
    ax[1].set_ylim([z_min-0.1, z_max + 0.1])
    ax[1].set_xlim([np.min(time), np.max(time)])
    ax[1].grid(True)

    # Initialize lines for both direct and iterative solver
    line_direct, = ax[0].plot([], [], '-', linewidth=4, label=f"Direct Solver, dt={dlo_position["DIRECT_SOLVER"]["timestep"]:.4f}")
    iter_lines = []
    for j in range(len(dlo_position["ITERATIVE_SOLVER"]["number_of_iterations"])):
        line, = ax[0].plot([], [], '-', linewidth=4, label=f"Nit={dlo_position["ITERATIVE_SOLVER"]["number_of_iterations"][j]}, dt={dlo_position["ITERATIVE_SOLVER"]["timestep"][j]:.4f}")
        iter_lines.append(line)
    ax[0].legend(loc="lower left")

    dz_line_direct, = ax[1].plot([], [], label=f"Direct Solver, dt={dlo_position["DIRECT_SOLVER"]["timestep"]:.4f}")
    dz_iter_lines = []
    for j in range(len(dlo_position["ITERATIVE_SOLVER"]["number_of_iterations"])):
        line, = ax[1].plot([], [], label=f"Nit={dlo_position["ITERATIVE_SOLVER"]["number_of_iterations"][j]}, dt={dlo_position["ITERATIVE_SOLVER"]["timestep"][j]:.4f}")
        dz_iter_lines.append(line)
    ax[1].legend(loc="upper right")

    def update(t):
        # DLO in xz-plane
        line_direct.set_data(
            dlo_position["DIRECT_SOLVER"]["x"][t, :],
            dlo_position["DIRECT_SOLVER"]["z"][t, :]
        )
        for j, line in enumerate(iter_lines):
            line.set_data(
                dlo_position["ITERATIVE_SOLVER"]["x"][j, t, :],
                dlo_position["ITERATIVE_SOLVER"]["z"][j, t, :]
            )
        
        # dz vs time
        dz_line_direct.set_data(
            time[:t],
            dz["DIRECT_SOLVER"][:t]  # tip node
        )
        for j, line in enumerate(dz_iter_lines):
            line.set_data(
                time[:t],
                dz["ITERATIVE_SOLVER"][j, :t]
            )

        return [line_direct, *iter_lines, dz_line_direct, *dz_iter_lines]

    # Display animation
    mpl.rcParams['animation.embed_limit'] = 200
    anim = FuncAnimation(fig, update, frames=len(time), interval=np.diff(time)[0]*1000, blit=True)
    plt.close(fig)

    return anim

anim = visualize_dlo(time, dlo_position, dlo_dz)
HTML(anim.to_jshtml())

In [ ]:
residual = []
for j in range(dlo_dz["ITERATIVE_SOLVER"].shape[0]):
    residual.append(np.linalg.norm(dlo_dz["DIRECT_SOLVER"] - dlo_dz["ITERATIVE_SOLVER"][j,:], ord=np.inf))

plt.figure()
plt.plot(dlo_position["ITERATIVE_SOLVER"]["number_of_iterations"], np.array(residual), "o-")
plt.xlabel("PGS-iterations")
plt.ylabel("Residual max-norm")
plt.show()


## Fix the number of PGS-iterations and vary the timestep size

In [ ]:
TIMESTEPS = [0.02, 0.01, 0.005]

# Ground truth from DENSE_LINEAR solver
settings_dict["timestep"] = TIMESTEPS[-1]
settings_dict["horizon"] = 2000
settings_dict["solver"] = Solver.DENSE_LINEAR
dlo_x, dlo_z, dz = simulate_dlo(settings_dict)

# To store results for plotting 
dlo_position = {"DIRECT_SOLVER": {"x": dlo_x, "z": dlo_z, "timestep": settings_dict["timestep"]}}
dlo_dz = {"DIRECT_SOLVER": dz}


# Vary the time step for PGS-solver
settings_dict["solver"] = Solver.SPARSE_PGS
settings_dict["pgs_iterations"] = 1000

smallest_dt = TIMESTEPS[-1]
time = np.array([k*smallest_dt for k in range(settings_dict["horizon"]+1)])
tmax = settings_dict["horizon"]*smallest_dt

dlo_pos_x = []
dlo_pos_z = []
delta_z = []

for dt in TIMESTEPS:
    
    settings_dict["timestep"] = dt
    this_time = np.array([k*dt for k in range(settings_dict["horizon"]+1) if k*dt <= tmax])
    dlo_x, dlo_z, dz = simulate_dlo(settings_dict, tmax=tmax)

    # To prepare data for visualization such they have equal length on time axis. Linear interpolation
    dlo_x = np.array([np.interp(time, this_time, dlo_x[:,j]) for j in range(dlo_x.shape[1])]).T
    dlo_z = np.array([np.interp(time, this_time, dlo_z[:,j]) for j in range(dlo_z.shape[1])]).T
    dz = np.interp(time, this_time, dz)

    dlo_pos_x.append(dlo_x)
    dlo_pos_z.append(dlo_z)
    delta_z.append(dz)

    jax.clear_caches()  # Workaround

# To store results for plotting
dlo_position["ITERATIVE_SOLVER"] = {"x": np.array(dlo_pos_x), "z": np.array(dlo_pos_z), "timestep": np.array(TIMESTEPS), "number_of_iterations": np.array([settings_dict["pgs_iterations"]]*len(TIMESTEPS))}
dlo_dz["ITERATIVE_SOLVER"] = np.array(delta_z)

anim = visualize_dlo(time, dlo_position, dlo_dz)
HTML(anim.to_jshtml())